# 02 - Data Profiling

## Objective

This notebook examines the structure and quality of the raw flight dataset before data cleaning and descriptive analytics.

The profiling process includes:

- Reviewing dataset dimensions and schema
- Measuring missing values
- Identifying duplicate records
- Summarizing numerical variables
- Reviewing key categorical variables
- Performing essential data-quality checks

The findings from this notebook will guide the data-cleaning and descriptive-analytics decisions.

#### Load configuration and dataset

In [1]:
import importlib.util
import os
from pathlib import Path

_bootstrap = Path.cwd() / "notebooks" / "import_path.py"
if not _bootstrap.is_file():
    _bootstrap = Path.cwd() / "import_path.py"
_spec = importlib.util.spec_from_file_location("import_path", _bootstrap)
_ip = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_ip)

# Load the project configuration and raw flight table

from config import project_config as cfg
from dotenv import dotenv_values
from pyspark.sql import functions as F
from pyspark.sql.types import NumericType

DATABRICKS_PROFILE = os.getenv(
    "DATABRICKS_CONFIG_PROFILE", "capstone-serverless"
)
api_env_candidates = [
    Path.cwd() / "api" / ".env",
    Path.cwd().parent / "api" / ".env",
]
api_env_path = next(
    (candidate for candidate in api_env_candidates if candidate.is_file()),
    None,
)
api_credentials = dotenv_values(api_env_path) if api_env_path else {}
LOCAL_DATABRICKS_HOST = str(
    api_credentials.get("DATABRICKS_SERVER_HOSTNAME", "")
).strip()
LOCAL_DATABRICKS_TOKEN = str(
    api_credentials.get("DATABRICKS_ACCESS_TOKEN", "")
).strip()
if LOCAL_DATABRICKS_HOST and not LOCAL_DATABRICKS_HOST.startswith("http"):
    LOCAL_DATABRICKS_HOST = f"https://{LOCAL_DATABRICKS_HOST}"

try:
    spark
except NameError:
    from databricks.connect import DatabricksSession

    builder = DatabricksSession.builder.serverless()
    if LOCAL_DATABRICKS_HOST and LOCAL_DATABRICKS_TOKEN:
        builder = builder.host(LOCAL_DATABRICKS_HOST).token(
            LOCAL_DATABRICKS_TOKEN
        )
    else:
        builder = builder.profile(DATABRICKS_PROFILE)
    spark = builder.getOrCreate()

if not spark.catalog.tableExists(cfg.RAW_TABLE):
    raise RuntimeError(
        f"Required table '{cfg.RAW_TABLE}' was not found. "
        "Run Notebook 01 before continuing."
    )

df_raw = spark.read.table(cfg.RAW_TABLE)

print(f"Table loaded: {cfg.RAW_TABLE}")

Table loaded: workspace.default.flights_raw


#### Dataset overview

This section reviews the size and structure of the raw flight dataset.

In [2]:
# Review dataset dimensions

row_count = df_raw.count()
column_count = len(df_raw.columns)

print(f"Total records: {row_count:,}")
print(f"Total columns: {column_count}")

Total records: 7,001,619
Total columns: 32


In [3]:
# Review the inferred schema

df_raw.printSchema()

root
 |-- QUARTER: integer (nullable = true)
 |-- MONTH: integer (nullable = true)
 |-- DAY_OF_WEEK: integer (nullable = true)
 |-- FL_DATE: string (nullable = true)
 |-- OP_UNIQUE_CARRIER: string (nullable = true)
 |-- OP_CARRIER_FL_NUM: integer (nullable = true)
 |-- ORIGIN: string (nullable = true)
 |-- ORIGIN_CITY_NAME: string (nullable = true)
 |-- ORIGIN_STATE_NM: string (nullable = true)
 |-- DEST: string (nullable = true)
 |-- DEST_CITY_NAME: string (nullable = true)
 |-- DEST_STATE_NM: string (nullable = true)
 |-- CRS_DEP_TIME: integer (nullable = true)
 |-- DEP_DELAY: double (nullable = true)
 |-- DEP_DEL15: double (nullable = true)
 |-- TAXI_OUT: double (nullable = true)
 |-- TAXI_IN: double (nullable = true)
 |-- CRS_ARR_TIME: integer (nullable = true)
 |-- ARR_DELAY: double (nullable = true)
 |-- ARR_DEL15: double (nullable = true)
 |-- CANCELLED: double (nullable = true)
 |-- CANCELLATION_CODE: string (nullable = true)
 |-- DIVERTED: double (nullable = true)
 |-- CRS_ELA

#### Missing-value analysis

Missing values are measured for every column to identify variables that may require removal, imputation, or special treatment during data cleaning.

In [4]:
# Calculate missing-value counts for every column

null_counts = (
    df_raw
    .select(
        [
            F.sum(
                F.col(column_name).isNull().cast("long")
            ).alias(column_name)
            for column_name in df_raw.columns
        ]
    )
    .first()
)

missing_values_data = [
    (
        field.name,
        field.dataType.simpleString(),
        int(null_counts[field.name] or 0),
        round(
            (null_counts[field.name] or 0)
            / row_count
            * 100,
            2
        ),
    )
    for field in df_raw.schema.fields
]

missing_values_df = spark.createDataFrame(
    missing_values_data,
    [
        "column_name",
        "data_type",
        "null_count",
        "null_percentage",
    ],
)

display(
    missing_values_df.orderBy(
        F.col("null_percentage").desc(),
        F.col("column_name"),
    )
)

,column_name,data_type,null_count,null_percentage
0,CANCELLATION_CODE,string,6898743,98.53
1,CARRIER_DELAY,double,5466981,78.08
2,LATE_AIRCRAFT_DELAY,double,5466981,78.08
3,NAS_DELAY,double,5466981,78.08
4,SECURITY_DELAY,double,5466981,78.08
5,WEATHER_DELAY,double,5466981,78.08
6,ACTUAL_ELAPSED_TIME,double,122135,1.74
7,AIR_TIME,double,122135,1.74
8,ARR_DEL15,double,122135,1.74
9,ARR_DELAY,double,122135,1.74


In [5]:
# Focus on the columns that actually contain missing values

display(
    missing_values_df
    .filter(F.col("null_count") > 0)
    .orderBy(F.col("null_percentage").desc())
)

,column_name,data_type,null_count,null_percentage
0,CANCELLATION_CODE,string,6898743,98.53
1,CARRIER_DELAY,double,5466981,78.08
2,WEATHER_DELAY,double,5466981,78.08
3,NAS_DELAY,double,5466981,78.08
4,SECURITY_DELAY,double,5466981,78.08
5,LATE_AIRCRAFT_DELAY,double,5466981,78.08
6,ARR_DELAY,double,122135,1.74
7,ARR_DEL15,double,122135,1.74
8,ACTUAL_ELAPSED_TIME,double,122135,1.74
9,AIR_TIME,double,122135,1.74


#### Missing-value interpretation

The highest null percentages are expected in `CANCELLATION_CODE` and the five delay-cause columns (`CARRIER_DELAY`, `WEATHER_DELAY`, `NAS_DELAY`, `SECURITY_DELAY`, `LATE_AIRCRAFT_DELAY`), followed by post-departure fields such as `DEP_DELAY`, `DEP_DEL15`, `TAXI_OUT`, `TAXI_IN`, `ARR_DELAY`, `ARR_DEL15`, `ACTUAL_ELAPSED_TIME`, and `AIR_TIME`.

This pattern is structural, not a data-quality defect:

- `CANCELLATION_CODE` is only populated for cancelled flights, so it is null for the vast majority of records.
- The delay-cause columns are only populated for flights with a recorded delay, so they are null whenever a flight was on time, cancelled, or diverted.
- `DEP_DELAY`, `TAXI_OUT`, `TAXI_IN`, `ARR_DELAY`, `ACTUAL_ELAPSED_TIME`, and `AIR_TIME` are unavailable for cancelled flights, since those flights never departed or landed.

These nulls are investigated formally, and cross-checked against cancellation and diversion status, in `03_data_cleaning`. Any column showing nulls outside this expected pattern would need further investigation before proceeding.

#### Duplicate-record analysis

This section identifies fully duplicated rows that could distort flight counts and analytical results.

In [6]:
# Calculate fully duplicated records

distinct_record_count = df_raw.distinct().count()
duplicate_record_count = row_count - distinct_record_count

duplicate_percentage = round(
    duplicate_record_count / row_count * 100,
    4
)

print(f"Distinct records: {distinct_record_count:,}")
print(f"Duplicate records: {duplicate_record_count:,}")
print(f"Duplicate percentage: {duplicate_percentage}%")

Distinct records: 7,001,619
Duplicate records: 0
Duplicate percentage: 0.0%


In [7]:
# Calculate records sharing the same business key (potential duplicate flight events)

distinct_business_key_count = (
    df_raw
    .select(*cfg.BUSINESS_KEY_COLUMNS)
    .distinct()
    .count()
)
business_key_duplicate_count = (
    row_count - distinct_business_key_count
)

business_key_duplicate_percentage = round(
    business_key_duplicate_count / row_count * 100,
    4
)

print(f"Distinct business keys: {distinct_business_key_count:,}")
print(
    f"Records sharing a business key: "
    f"{business_key_duplicate_count:,}"
)
print(
    f"Business-key duplicate percentage: "
    f"{business_key_duplicate_percentage}%"
)

Distinct business keys: 7,001,619
Records sharing a business key: 0
Business-key duplicate percentage: 0.0%


#### Numerical summary

This section summarizes the distribution of numerical variables, including their central tendency, variation, ranges, and quartiles.

In [8]:
# Identify numerical columns

numeric_columns = [
    field.name
    for field in df_raw.schema.fields
    if isinstance(field.dataType, NumericType)
]

print(f"Numerical columns: {len(numeric_columns)}")

Numerical columns: 23


In [9]:
# Display descriptive statistics for numerical columns

display(
    df_raw
    .select(numeric_columns)
    .summary(
        "count",
        "mean",
        "stddev",
        "min",
        "25%",
        "50%",
        "75%",
        "max",
    )
)

,summary,QUARTER,MONTH,DAY_OF_WEEK,OP_CARRIER_FL_NUM,CRS_DEP_TIME,DEP_DELAY,DEP_DEL15,TAXI_OUT,TAXI_IN,CRS_ARR_TIME,ARR_DELAY,ARR_DEL15,CANCELLED,DIVERTED,CRS_ELAPSED_TIME,ACTUAL_ELAPSED_TIME,AIR_TIME,DISTANCE,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY
0,count,7001619,7001619,7001619,7001619,7001619,6903028,6903028,6899436,6897011,7001619,6879484,6879484,7001619,7001619,7001617,6879484,6879484,7001619,1534638,1534638,1534638,1534638,1534638
1,mean,2.5238985440367436,6.571155899799746,3.999110062972578,2517.2882784681656,1323.9459002267904,13.55640669572831,0.21756032280326837,18.64623238769082,8.611399053880007,1492.304588695843,8.504504407598011,0.2230745794306666,0.01469317310753413,0.0027505067042351205,148.9098054063797,144.08510042322942,116.84434065113022,844.4473882397771,23.667129968109744,4.632176448126529,15.95794317617575,0.09956875823484106,28.581965909875816
2,stddev,1.1053655450121078,3.3941441738648632,2.0105750716788218,1632.1071649054168,492.21964345817145,57.53803565682585,0.41258678287711525,10.690151758679919,7.352404748693689,518.5739720584326,59.73557564368515,0.41630798291064974,0.12032159340397561,0.05237310195952977,72.95441364646462,73.25642635293599,71.25512780390785,601.9752060669634,74.7364602002375,34.81616637943106,36.43684068395454,3.336252368528487,61.76801607785079
3,min,1,1,1,1,1,-115.0,0.0,1.0,1.0,1,-128.0,0.0,0.0,0.0,-99.0,15.0,7.0,31.0,0.0,0.0,0.0,0.0,0.0
4,25%,2,4,2,1197,904,-6.0,0.0,12.0,5.0,1103,-15.0,0.0,0.0,0.0,95.0,90.0,64.0,403.0,0.0,0.0,0.0,0.0,0.0
5,50%,3,7,4,2270,1318,-2.0,0.0,16.0,7.0,1518,-6.0,0.0,0.0,0.0,132.0,128.0,100.0,693.0,2.0,0.0,1.0,0.0,1.0
6,75%,4,10,6,3645,1735,10.0,0.0,21.0,10.0,1926,11.0,0.0,0.0,0.0,180.0,176.0,147.0,1080.0,20.0,0.0,19.0,0.0,33.0
7,max,4,12,7,9914,2400,4352.0,1.0,1274.0,1318.0,2400,4336.0,1.0,1.0,1.0,1510.0,965.0,953.0,5095.0,4336.0,2394.0,1706.0,990.0,2425.0


#### Key categorical variables

This section reviews the cardinality and most frequent values of the main categorical variables used in the analysis.

In [10]:
# Keep only configured categorical columns available in the dataset

categorical_columns = [
    column_name
    for column_name in cfg.KEY_CATEGORICAL_COLUMNS
    if column_name in df_raw.columns
]

categorical_summary = []

for column_name in categorical_columns:
    distinct_count = (
        df_raw
        .select(column_name)
        .distinct()
        .count()
    )

    categorical_summary.append(
        (column_name, distinct_count)
    )

categorical_summary_df = spark.createDataFrame(
    categorical_summary,
    [
        "column_name",
        "distinct_value_count",
    ],
)

display(
    categorical_summary_df.orderBy(
        F.col("distinct_value_count").desc()
    )
)

,column_name,distinct_value_count
0,ORIGIN,352
1,DEST,352
2,OP_UNIQUE_CARRIER,14
3,CANCELLATION_CODE,5


In [11]:
# Display the most frequent values for key categorical columns

for column_name in categorical_columns:
    print(f"Most frequent values for {column_name}:")

    display(
        df_raw
        .groupBy(column_name)
        .count()
        .orderBy(F.col("count").desc())
        .limit(cfg.TOP_N_RESULTS)
    )

Most frequent values for OP_UNIQUE_CARRIER:


,OP_UNIQUE_CARRIER,count
0,WN,1391885
1,DL,1026332
2,AA,973653
3,OO,839821
4,UA,795271
5,YX,346036
6,MQ,299322
7,OH,248735
8,AS,245588
9,B6,231413


Most frequent values for ORIGIN:


,ORIGIN,count
0,ORD,327028
1,DEN,317686
2,DFW,315854
3,ATL,314433
4,PHX,196756
5,CLT,195425
6,LAX,190472
7,LAS,183803
8,SEA,164835
9,MCO,160048


Most frequent values for DEST:


,DEST,count
0,ORD,327011
1,DEN,317675
2,DFW,315852
3,ATL,314439
4,PHX,196757
5,CLT,195428
6,LAX,190499
7,LAS,183787
8,SEA,164834
9,MCO,160000


Most frequent values for CANCELLATION_CODE:


,CANCELLATION_CODE,count
0,None,6898743
1,B,63404
2,A,20903
3,C,18519
4,D,50


#### Essential data-quality checks

This section verifies the valid ranges and values of the main operational variables.

In [12]:
# Perform essential range and binary-value checks

quality_checks = [
    (
        "Invalid QUARTER values",
        df_raw.filter(
            F.col(cfg.QUARTER_COLUMN).isNull()
            | ~F.col(cfg.QUARTER_COLUMN).between(1, 4)
        ).count(),
    ),
    (
        "Invalid MONTH values",
        df_raw.filter(
            F.col(cfg.MONTH_COLUMN).isNull()
            | ~F.col(cfg.MONTH_COLUMN).between(1, 12)
        ).count(),
    ),
    (
        "Invalid DAY_OF_WEEK values",
        df_raw.filter(
            F.col(cfg.DAY_OF_WEEK_COLUMN).isNull()
            | ~F.col(cfg.DAY_OF_WEEK_COLUMN).between(1, 7)
        ).count(),
    ),
    (
        "Invalid CANCELLED values",
        df_raw.filter(
            F.col(cfg.CANCELLED_COLUMN).isNull()
            | ~F.col(cfg.CANCELLED_COLUMN).isin(0, 1)
        ).count(),
    ),
    (
        "Invalid DIVERTED values",
        df_raw.filter(
            F.col(cfg.DIVERTED_COLUMN).isNull()
            | ~F.col(cfg.DIVERTED_COLUMN).isin(0, 1)
        ).count(),
    ),
    (
        "Invalid ARR_DEL15 values",
        df_raw.filter(
            F.col(cfg.ARRIVAL_DELAY_FLAG_COLUMN).isNotNull()
            & ~F.col(
                cfg.ARRIVAL_DELAY_FLAG_COLUMN
            ).isin(0, 1)
        ).count(),
    ),
    (
        "Invalid DEP_DEL15 values",
        df_raw.filter(
            F.col(cfg.DEPARTURE_DELAY_FLAG_COLUMN).isNotNull()
            & ~F.col(
                cfg.DEPARTURE_DELAY_FLAG_COLUMN
            ).isin(0, 1)
        ).count(),
    ),
    (
        "Non-positive DISTANCE values",
        df_raw.filter(
            F.col(cfg.DISTANCE_COLUMN).isNull()
            | (F.col(cfg.DISTANCE_COLUMN) <= 0)
        ).count(),
    ),
    (
        "Non-positive CRS_ELAPSED_TIME values",
        df_raw.filter(
            F.col(cfg.SCHEDULED_ELAPSED_TIME_COLUMN).isNull()
            | (F.col(cfg.SCHEDULED_ELAPSED_TIME_COLUMN) <= 0)
        ).count(),
    ),
    (
        "Non-positive ACTUAL_ELAPSED_TIME values",
        df_raw.filter(
            F.col(cfg.ACTUAL_ELAPSED_TIME_COLUMN).isNotNull()
            & (F.col(cfg.ACTUAL_ELAPSED_TIME_COLUMN) <= 0)
        ).count(),
    ),
    (
        "Non-positive AIR_TIME values",
        df_raw.filter(
            F.col(cfg.AIR_TIME_COLUMN).isNotNull()
            & (F.col(cfg.AIR_TIME_COLUMN) <= 0)
        ).count(),
    ),
    (
        "Negative TAXI_OUT values",
        df_raw.filter(
            F.col(cfg.TAXI_OUT_COLUMN).isNotNull()
            & (F.col(cfg.TAXI_OUT_COLUMN) < 0)
        ).count(),
    ),
    (
        "Negative TAXI_IN values",
        df_raw.filter(
            F.col(cfg.TAXI_IN_COLUMN).isNotNull()
            & (F.col(cfg.TAXI_IN_COLUMN) < 0)
        ).count(),
    ),
]

# Extend with a negative-value check for each delay-cause column
# (null is tolerated: delay-cause columns are only populated when a delay occurred)

quality_checks.extend(
    [
        (
            f"Negative {column_name} values",
            df_raw.filter(
                F.col(column_name).isNotNull()
                & (F.col(column_name) < 0)
            ).count(),
        )
        for column_name in cfg.DELAY_CAUSE_COLUMNS
    ]
)

quality_checks_df = spark.createDataFrame(
    quality_checks,
    [
        "quality_check",
        "invalid_record_count",
    ],
)

display(quality_checks_df)

,quality_check,invalid_record_count
0,Invalid QUARTER values,0
1,Invalid MONTH values,0
2,Invalid DAY_OF_WEEK values,0
3,Invalid CANCELLED values,0
4,Invalid DIVERTED values,0
5,Invalid ARR_DEL15 values,0
6,Invalid DEP_DEL15 values,0
7,Non-positive DISTANCE values,0
8,Non-positive CRS_ELAPSED_TIME values,7
9,Non-positive ACTUAL_ELAPSED_TIME values,0


#### Dataset preview

A small sample is displayed to support a final visual review of the raw records.

In [13]:
display(df_raw.limit(10))

,QUARTER,MONTH,DAY_OF_WEEK,FL_DATE,OP_UNIQUE_CARRIER,OP_CARRIER_FL_NUM,ORIGIN,ORIGIN_CITY_NAME,ORIGIN_STATE_NM,DEST,DEST_CITY_NAME,DEST_STATE_NM,CRS_DEP_TIME,DEP_DELAY,DEP_DEL15,TAXI_OUT,TAXI_IN,CRS_ARR_TIME,ARR_DELAY,ARR_DEL15,CANCELLED,CANCELLATION_CODE,DIVERTED,CRS_ELAPSED_TIME,ACTUAL_ELAPSED_TIME,AIR_TIME,DISTANCE,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY
0,3,9,5,9/12/2025 12:00:00 AM,DL,406,BZN,"Bozeman, MT",Montana,ATL,"Atlanta, GA",Georgia,1419,-4.0,0.0,9.0,9.0,2011,-27.0,0.0,0.0,None,0.0,232.0,209.0,191.0,1640.0,NaN,NaN,NaN,NaN,NaN
1,3,9,5,9/12/2025 12:00:00 AM,DL,407,LAX,"Los Angeles, CA",California,DTW,"Detroit, MI",Michigan,2245,23.0,1.0,16.0,9.0,600,20.0,1.0,0.0,None,0.0,255.0,252.0,227.0,1979.0,0.0,0.0,0.0,0.0,20.0
2,3,9,5,9/12/2025 12:00:00 AM,DL,408,LAS,"Las Vegas, NV",Nevada,BOS,"Boston, MA",Massachusetts,1115,-12.0,0.0,35.0,13.0,1934,-13.0,0.0,0.0,None,0.0,319.0,318.0,270.0,2381.0,NaN,NaN,NaN,NaN,NaN
3,3,9,5,9/12/2025 12:00:00 AM,DL,409,JFK,"New York, NY",New York,SFO,"San Francisco, CA",California,900,-1.0,0.0,42.0,26.0,1221,2.0,0.0,0.0,None,0.0,381.0,384.0,316.0,2586.0,NaN,NaN,NaN,NaN,NaN
4,3,9,5,9/12/2025 12:00:00 AM,DL,410,DCA,"Washington, DC",Virginia,ATL,"Atlanta, GA",Georgia,600,-6.0,0.0,11.0,8.0,746,-13.0,0.0,0.0,None,0.0,106.0,99.0,80.0,547.0,NaN,NaN,NaN,NaN,NaN
5,3,9,5,9/12/2025 12:00:00 AM,DL,411,MSP,"Minneapolis, MN",Minnesota,ANC,"Anchorage, AK",Alaska,2245,0.0,0.0,13.0,7.0,136,-15.0,0.0,0.0,None,0.0,351.0,336.0,316.0,2519.0,NaN,NaN,NaN,NaN,NaN
6,3,9,5,9/12/2025 12:00:00 AM,DL,412,GSP,"Greer, SC",South Carolina,DTW,"Detroit, MI",Michigan,705,-7.0,0.0,11.0,15.0,852,-17.0,0.0,0.0,None,0.0,107.0,97.0,71.0,508.0,NaN,NaN,NaN,NaN,NaN
7,3,9,5,9/12/2025 12:00:00 AM,DL,413,LAX,"Los Angeles, CA",California,MSP,"Minneapolis, MN",Minnesota,1125,21.0,1.0,18.0,9.0,1702,22.0,1.0,0.0,None,0.0,217.0,218.0,191.0,1535.0,8.0,0.0,1.0,0.0,13.0
8,3,9,5,9/12/2025 12:00:00 AM,DL,414,ATL,"Atlanta, GA",Georgia,DFW,"Dallas/Fort Worth, TX",Texas,1130,53.0,1.0,9.0,9.0,1242,46.0,1.0,0.0,None,0.0,132.0,125.0,107.0,731.0,34.0,0.0,0.0,0.0,12.0
9,3,9,5,9/12/2025 12:00:00 AM,DL,414,DFW,"Dallas/Fort Worth, TX",Texas,ATL,"Atlanta, GA",Georgia,1347,44.0,1.0,13.0,8.0,1658,30.0,1.0,0.0,None,0.0,131.0,117.0,96.0,731.0,0.0,0.0,0.0,0.0,30.0


#### Profiling completion

In [14]:
print("Data profiling completed successfully.")

Data profiling completed successfully.
